# 02 — Simulation Validation

Validate the simulation engine by checking:
1. CDM mass function slope against theory (-1.9)
2. WDM/FDM suppression relative to CDM
3. Gap formation from subhalo impulse kicks
4. Gaia noise injection realism
5. Simulation dataset statistics

In [ ]:
import sys
from pathlib import Path
import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

from src.simulation.mass_functions import (
    sample_cdm_masses, sample_wdm_masses, sample_fdm_masses,
    wdm_half_mode_mass, fdm_jeans_mass
)
from src.simulation.noise import gaia_pm_uncertainty, gaia_parallax_uncertainty

## Mass function comparison

In [ ]:
n = 20000
cdm = sample_cdm_masses(n, seed=42)
wdm = sample_wdm_masses(n, m_wdm_kev=3.0, seed=42)
fdm = sample_fdm_masses(n, m_axion_ev=1e-22, seed=42)

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.logspace(5, 9, 50)
ax.hist(cdm, bins=bins, alpha=0.5, label='CDM', density=True)
ax.hist(wdm, bins=bins, alpha=0.5, label='WDM (3 keV)', density=True)
ax.hist(fdm, bins=bins, alpha=0.5, label='FDM (m22=1)', density=True)
ax.set_xscale('log')
ax.set_xlabel('Subhalo mass [Msun]')
ax.set_ylabel('Normalized count')
ax.set_title('Mass function comparison')
ax.legend()

# Mark half-mode / Jeans masses
m_hm = wdm_half_mode_mass(3.0)
m_j = fdm_jeans_mass(1e-22)
ax.axvline(m_hm, color='orange', ls='--', label=f'WDM M_hm = {m_hm:.1e}')
ax.axvline(m_j, color='green', ls='--', label=f'FDM M_J = {m_j:.1e}')
ax.legend()
plt.tight_layout()
plt.show()

## CDM slope measurement

In [ ]:
masses = sample_cdm_masses(50000, log10_m_min=6.0, log10_m_max=9.0, alpha=-1.9, seed=0)
log_masses = np.log10(masses)
counts, edges = np.histogram(log_masses, bins=30)
centers = 0.5 * (edges[:-1] + edges[1:])
valid = counts > 10

slope, intercept = np.polyfit(centers[valid], np.log10(counts[valid] + 1), 1)
print(f'Measured slope: {slope:.3f} (expected: -0.9 for alpha=-1.9)')
print(f'Offset from theory: {abs(slope + 0.9):.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(centers[valid], np.log10(counts[valid] + 1), s=15)
ax.plot(centers, intercept + slope * centers, 'r--', label=f'Fit: slope={slope:.2f}')
ax.set_xlabel('log10(M/Msun)')
ax.set_ylabel('log10(count + 1)')
ax.set_title('CDM mass function slope validation')
ax.legend()
plt.tight_layout()
plt.show()

## Gaia noise model

In [ ]:
g_mag = np.linspace(13, 21, 100)
sigma_pm = gaia_pm_uncertainty(g_mag)
sigma_plx = gaia_parallax_uncertainty(g_mag)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(g_mag, sigma_pm)
ax1.set_xlabel('G magnitude')
ax1.set_ylabel('sigma_pm [mas/yr]')
ax1.set_title('Gaia PM uncertainty model')
ax1.axhline(0.1, color='gray', ls=':', label='0.1 mas/yr')
ax1.legend()

ax2.semilogy(g_mag, sigma_plx)
ax2.set_xlabel('G magnitude')
ax2.set_ylabel('sigma_plx [mas]')
ax2.set_title('Gaia parallax uncertainty model')
plt.tight_layout()
plt.show()

## Simulation dataset overview

In [ ]:
sim_dir = ROOT / 'data' / 'simulations'
dm_counts = {'CDM': 0, 'WDM': 0, 'FDM': 0, 'SIDM': 0}
stream_counts = {}
total = 0

for h5_path in sorted(sim_dir.glob('**/*.h5')):
    with h5py.File(str(h5_path), 'r') as f:
        if 'simulations' not in f:
            continue
        for run_id in f['simulations'].keys():
            grp = f[f'simulations/{run_id}']
            dm = grp.attrs.get('dm_model', b'unknown')
            if isinstance(dm, bytes):
                dm = dm.decode()
            sn = grp.attrs.get('stream_name', b'unknown')
            if isinstance(sn, bytes):
                sn = sn.decode()
            dm_counts[dm] = dm_counts.get(dm, 0) + 1
            stream_counts[sn] = stream_counts.get(sn, 0) + 1
            total += 1

print(f'Total simulations: {total}')
print(f'\nBy DM model:')
for m, c in sorted(dm_counts.items()):
    print(f'  {m:6s}: {c}')
print(f'\nBy stream:')
for s, c in sorted(stream_counts.items()):
    print(f'  {s:8s}: {c}')